# QQA 13 – Typed primal–dual production runtime

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Yuma-Ichikawa/QQA4CO/blob/main/examples/13_typed_primal_dual_runtime.ipynb)

Diagnose, solve, visualize, resume, and package a typed optimization run.

In [ ]:
# Install QQA on Google Colab (no-op if already installed locally).
# We prefer the released wheel on PyPI; users who want bleeding-edge
# ``main`` can set QQA_INSTALL_FROM_GIT=1 before running this cell.
import importlib.util
import os
import subprocess
import sys

if importlib.util.find_spec("qqa") is None:
    if os.environ.get("QQA_INSTALL_FROM_GIT") == "1":
        spec = "qqa @ git+https://github.com/Yuma-Ichikawa/QQA4CO.git"
    else:
        spec = "qqa"
    subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", spec])

## What this notebook verifies

This walkthrough exercises the strict typed runtime: model diagnosis, goal/budget solving, tri-state feasibility, schema-v2 events, the optimization cockpit, pickle-free checkpoint/resume, and a verified result package. It needs no API key and writes only to a temporary directory that is deleted automatically.

In [ ]:
import tempfile
from pathlib import Path

import networkx as nx

import qqa
from qqa.runtime import (
    export_result_package,
    fingerprint_problem,
    verify_result_package,
)
from qqa.visuals import decision_explorer, plot_optimization_cockpit

qqa.fix_seed(0)
print("QQA version:", qqa.__version__)

## 1. Build and diagnose a typed model

The facility-location template preserves assignment constraints. The doctor checks bounds, capabilities, scaling, decomposition, routes, and estimated resources without running a solver.

In [ ]:
model = qqa.build_template(
    "facility-location",
    opening_costs=[4.0, 5.0, 3.0],
    assignment_costs=[[1.0, 4.0, 2.0], [3.0, 1.0, 2.0]],
)
doctor = qqa.doctor(model, replicas=32)
print(doctor.explain())
doctor.to_dict()["solver_routes"]

## 2. Solve by outcome and duration

Unknown feasibility is never displayed as feasible. Exact proof is disabled in this short CPU example; use `goal="prove"` with a compatible exact extra when a certificate is required.

In [ ]:
result = qqa.solve(
    model,
    goal="feasible",
    budget="3s",
    device="cpu",
    replicas=32,
    epochs=80,
    exact_backend="none",
    seed=0,
)
{
    "status": result.status.value,
    "feasibility": result.violations.status.value,
    "objective": result.best_obj,
    "events": len(result.events),
}

## 3. Inspect dynamics and decisions

The cockpit uses the same result/event contract as other backends. The decision table reports archive stability and binary counterfactuals.

In [ ]:
figure, _ = plot_optimization_cockpit(result, backend="matplotlib")
figure.show()
decision_explorer(result, model)[:5]

## 4. Resume and verify an exchange package

Checkpoints contain JSON plus checksum-protected NumPy tensors—never pickle. The model/config fingerprint, RNG, optimizer, schedule, population, incumbent, and archive are validated before continuation.

In [ ]:
problem = qqa.MaxCut(nx.cycle_graph(8))
with tempfile.TemporaryDirectory() as temporary:
    root = Path(temporary)
    checkpoint = root / "state.qqacp"
    package = root / "result.qqapkg"
    qqa.solve(
        problem,
        profile="fast",
        replicas=16,
        epochs=4,
        exact_backend="none",
        checkpoint_path=checkpoint,
        checkpoint_interval=2,
        seed=7,
    )
    resumed = qqa.solve(
        problem,
        profile="fast",
        replicas=16,
        epochs=8,
        exact_backend="none",
        resume_from=checkpoint,
        seed=7,
    )
    export_result_package(
        resumed,
        package,
        model_summary={"name": "cycle-maxcut", "variables": 8},
        model_fingerprint=fingerprint_problem(problem),
    )
    manifest = verify_result_package(package)
    print("resumed epochs:", resumed.diagnostics["completed_epochs"])
    print("verified package schema:", manifest.schema_version)

## Production checklist

- Use typed JSON ModelIR for untrusted/remote jobs.
- Keep Python source and pickle disabled on shared services.
- Record the package checksum, benchmark snapshot hash, seed, budget, and exact backend.
- Treat `solver-reported-*` certificate metadata as a solver claim unless an independent verifier and proof digest are present.
- Validate the returned solution in original space and high precision.